# COVID-Only vs UKR-Only Eval Results

Reads `covid_ukr_only_eval_results_0_3_10_2026_06_15.csv` from this directory and plots the two independently trained checkpoints across the graph/task eval suite.

This notebook intentionally validates provenance before plotting: only eval runs from `15_06_2026`, only `0/3/10` shots, and only `covid_only_nm` / `ukr_only_nm` model families are allowed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CSV_NAME = "covid_ukr_only_eval_results_0_3_10_2026_06_15.csv"
CSV_PATH = Path(CSV_NAME)
if not CSV_PATH.exists():
    CSV_PATH = Path("scripts/plotting/covid_only_ukr_only") / CSV_NAME
CSV_PATH = CSV_PATH.resolve()

# Swap these when needed.
SPLIT = "test"
METRIC = "roc_auc"

EVAL_DATE = "15_06_2026"
SHOT_ORDER = [0, 3, 10]

DATASET_ORDER = [
    "covid19_twitter",
    "ukr_rus_twitter",
    "midterm",
    "covid_political",
    "election2020",
    "ukr_rus_suspended",
]

TASK_ORDER = ["nm", "lp", "pl"]
TASK_LABELS = {
    "nm": "Neighbor matching",
    "lp": "Temporal link prediction",
    "pl": "Classification",
}
MODEL_ORDER = ["covid_only_nm", "ukr_only_nm"]
MODEL_LABELS = {
    "covid_only_nm": "COVID-only NM",
    "ukr_only_nm": "UKR-only NM",
}
METRICS = ["accuracy", "f1", "roc_auc"]

In [ ]:
df = pd.read_csv(CSV_PATH)
df["shots"] = df["shots"].astype(int)

for col in METRICS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["model_family"] = df["model"].str.extract(r"^(covid_only_nm|ukr_only_nm)", expand=False)
df["model_label"] = df["model_family"].map(MODEL_LABELS).fillna(df["model"])
df["eval_date"] = df["timestamp"].astype(str).str[:10]

unexpected_models = sorted(set(df["model_family"].dropna()) ^ set(MODEL_ORDER))
missing_model_family = df[df["model_family"].isna()]["model"].drop_duplicates().tolist()
unexpected_dates = sorted(set(df["eval_date"]) - {EVAL_DATE})
unexpected_shots = sorted(set(df["shots"]) - set(SHOT_ORDER))

if missing_model_family:
    raise ValueError(f"Unexpected model names: {missing_model_family}")
if unexpected_models:
    raise ValueError(f"Expected exactly {MODEL_ORDER}; mismatch: {unexpected_models}")
if unexpected_dates:
    raise ValueError(f"Found eval dates outside {EVAL_DATE}: {unexpected_dates}")
if unexpected_shots:
    raise ValueError(f"Found shots outside {SHOT_ORDER}: {unexpected_shots}")

df["dataset"] = pd.Categorical(df["dataset"], categories=DATASET_ORDER, ordered=True)
df["task"] = pd.Categorical(df["task"], categories=TASK_ORDER, ordered=True)
df["model_family"] = pd.Categorical(df["model_family"], categories=MODEL_ORDER, ordered=True)
df["task_label"] = df["task"].astype(str).map(TASK_LABELS).fillna(df["task"].astype(str))
df = df.sort_values(["split", "dataset", "task", "model_family", "shots"])

print(f"reading {CSV_PATH}")
print(f"rows: {len(df)}")
display(df.groupby(["model_family", "shots"], observed=True).size().rename("rows").reset_index())
display(df.groupby(["task", "shots"], observed=True).size().rename("rows").reset_index())
display(df.head())
display(
    df.groupby(["split", "dataset", "task", "model_family"], observed=True)
      .agg(shots=("shots", lambda x: sorted(set(x))), rows=("shots", "size"))
      .reset_index()
)

In [ ]:
def plot_metric_grid(data, *, split=SPLIT, metric=METRIC, datasets=DATASET_ORDER, tasks=TASK_ORDER):
    subset = data[(data["split"] == split) & data[metric].notna()].copy()
    if subset.empty:
        raise ValueError(f"No rows for split={split!r}, metric={metric!r}")

    palette = dict(zip(MODEL_ORDER, sns.color_palette("Set2", n_colors=len(MODEL_ORDER))))

    fig, axes = plt.subplots(
        nrows=len(datasets),
        ncols=len(tasks),
        figsize=(4.6 * len(tasks), 2.7 * len(datasets)),
        sharex=True,
        sharey=True,
    )

    if len(datasets) == 1 and len(tasks) == 1:
        axes = [[axes]]
    elif len(datasets) == 1:
        axes = [axes]
    elif len(tasks) == 1:
        axes = [[ax] for ax in axes]

    for row_idx, dataset in enumerate(datasets):
        for col_idx, task in enumerate(tasks):
            ax = axes[row_idx][col_idx]
            panel = subset[(subset["dataset"].astype(str) == dataset) & (subset["task"].astype(str) == task)]
            task_label = TASK_LABELS.get(task, task)
            ax.set_title(f"{dataset}\n{task_label}", fontsize=10)

            if panel.empty:
                ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center", va="center", color="0.45")
            else:
                for model_family in MODEL_ORDER:
                    line = panel[panel["model_family"].astype(str) == model_family].sort_values("shots")
                    if line.empty:
                        continue
                    ax.plot(
                        line["shots"],
                        line[metric],
                        marker="o",
                        linewidth=2,
                        markersize=5,
                        label=MODEL_LABELS.get(model_family, model_family),
                        color=palette[model_family],
                    )

            ax.set_ylim(0, 1.02)
            ax.set_xticks(SHOT_ORDER)
            ax.grid(True, alpha=0.3)
            if row_idx == len(datasets) - 1:
                ax.set_xlabel("shots")
            else:
                ax.set_xlabel("")
            if col_idx == 0:
                ax.set_ylabel(metric)
            else:
                ax.set_ylabel("")

    handles, labels = [], []
    for ax in fig.axes:
        h, l = ax.get_legend_handles_labels()
        for handle, label in zip(h, l):
            if label not in labels:
                handles.append(handle)
                labels.append(label)

    if handles:
        fig.legend(handles, labels, title="trained model", loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.01))

    fig.suptitle(f"{split} {metric} by graph, task, shots, and training graph", y=1.03, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_metric_grid(df, split=SPLIT, metric=METRIC)

In [ ]:
# Side-by-side table for the currently selected split/metric.
table = (
    df[df["split"] == SPLIT]
    .pivot_table(
        index=["dataset", "task", "shots"],
        columns="model_label",
        values=METRIC,
        observed=True,
    )
    .reset_index()
    .sort_values(["dataset", "task", "shots"])
)

model_cols = [MODEL_LABELS[m] for m in MODEL_ORDER if MODEL_LABELS[m] in table.columns]
if len(model_cols) == 2:
    table[f"delta ({model_cols[1]} - {model_cols[0]})"] = table[model_cols[1]] - table[model_cols[0]]

display(table)

In [ ]:
# Compact long-form table for export/copying.
long_table = (
    df[df["split"] == SPLIT]
    [["dataset", "task", "model_label", "shots", METRIC, "run_name"]]
    .sort_values(["dataset", "task", "model_label", "shots"])
)
display(long_table)